# Day 1 / Session 1: Modern LLM Foundations -- Demos

This notebook contains all instructor-led demonstrations for Session 1.
Run each cell, observe the output, and make sure the concepts are clear
before moving on to the exercises notebook (`D1S1_exercises.ipynb`).

**What you will see:**
1. Setting up LLM API connections
2. Tokenization and context windows
3. Controlling model output with parameters
4. Shaping behavior with system messages
5. Chaining LLM calls into a pipeline
6. Embeddings and vector representations (narrative)
7. How LLMs support reasoning and agent behavior

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# ============================================================
# Imports and Setup
# ============================================================

import openai
import tiktoken
import os
import json
import numpy as np

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or set it in a .env file in the project root.

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demo 1: Setting Up LLM API Connections

In this demo we initialize the OpenAI client and make our first chat completion call.
The `client` object handles authentication, request formatting, and response parsing.

**Scenario:** You are building a tool that lets a consulting team quickly query an LLM
to summarize industry concepts for a client engagement kickoff.

In [ ]:
# Demo 1 - Setting Up LLM API Connections

# Step 1: Initialize the OpenAI client
# The client automatically reads OPENAI_API_KEY from the environment.
client = openai.OpenAI()

# Step 2: Make a simple chat completion call
# Scenario: A consulting analyst needs a quick definition for a client deck
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "What is a digital transformation strategy? Explain in two sentences for a McKinsey client presentation."}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)

# Step 3: Print the response
print("Model used :", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\nAssistant response:")
print(response.choices[0].message.content)

---
## Demo 2: Understanding Tokenization and Context Windows

Tokens are the fundamental units that LLMs process. A token can be a word, part of a
word, or even a single character. Understanding tokenization helps you:
- Estimate API costs (billing is per-token)
- Stay within context window limits
- Write more efficient prompts

In [ ]:
# Demo 2 - Understanding Tokenization and Context Windows

# Step 1: Get the tokenizer for gpt-4o-mini
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

# Step 2: Encode various consulting-related texts and count tokens
texts = [
    "McKinsey & Company",
    "Digital transformation is reshaping how Fortune 500 companies create value across their organizations.",
    "The client's operating model requires restructuring to achieve the target $200M in synergies.",
    "Organizational restructuring and operational efficiency optimization",
    "def calculate_market_size(tam, penetration_rate):\n    sam = tam * penetration_rate\n    return sam * 0.15  # McKinsey target share"
]

print("Token counts for different consulting texts:")
print("=" * 60)
for text in texts:
    tokens = encoder.encode(text)
    print(f"\nText   : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Chars  : {len(text)}")
    print(f"Tokens : {len(tokens)}")
    print(f"Token IDs (first 10): {tokens[:10]}")

# Step 3: Demonstrate context window limits
print("\n" + "=" * 60)
print("Context window reference:")
print("  gpt-4o-mini : 128,000 tokens")
print("  gpt-4o      : 128,000 tokens")
print("  gpt-3.5-turbo: 16,385 tokens")

# Simulate a long due diligence report consuming the window
long_text = "The target company operates in a fragmented market with strong growth potential. " * 500
long_tokens = encoder.encode(long_text)
print(f"\nLong due diligence excerpt ({len(long_text)} chars) = {len(long_tokens)} tokens")
print(f"That is {len(long_tokens)/128000*100:.2f}% of the gpt-4o-mini context window.")

---
## Demo 3: Exploring Model Parameters

LLMs expose several parameters that control the randomness and length of generated text:

| Parameter | Description |
|-----------|-------------|
| `temperature` | Controls randomness. 0 = deterministic, 1 = creative |
| `top_p` | Nucleus sampling. Lower values = more focused output |
| `max_tokens` | Maximum number of tokens in the response |

In [12]:
# Demo 3 - Exploring Model Parameters

client = openai.OpenAI()
prompt = "Write a one-sentence executive summary of generative AI's impact on the management consulting industry."

# --- Temperature comparison ---
print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0.0, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\nTemperature {temp}:")
    print(f"  {response.choices[0].message.content}")

# --- top_p comparison ---
print("\n" + "=" * 60)
print("TOP_P COMPARISON")
print("=" * 60)

for top_p in [0.1, 0.5, 1.0]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        top_p=top_p,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80"))
    )
    print(f"\ntop_p {top_p}:")
    print(f"  {response.choices[0].message.content}")

# --- max_tokens comparison ---
print("\n" + "=" * 60)
print("MAX_TOKENS COMPARISON")
print("=" * 60)

for max_tok in [10, 30, 100]:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")),
        max_tokens=max_tok
    )
    text = response.choices[0].message.content
    print(f"\nmax_tokens={max_tok} (finish_reason={response.choices[0].finish_reason}):")
    print(f"  {text}")


max_tokens=100 (finish_reason=stop):
  Generative AI is revolutionizing the management consulting industry by enhancing data analysis, streamlining report generation, and enabling more personalized client solutions, ultimately driving efficiency and innovation in service delivery.


---
## Demo 4: Comparing LLM Response Behaviors

The **system message** is a powerful lever for controlling how the LLM behaves.
By changing the system message, you can make the same model act as different expert
"personas," each with its own tone, expertise, and communication style -- mirroring
how different practice areas approach the same client question.

In [13]:
# Demo 4 - Comparing LLM Response Behaviors

client = openai.OpenAI()

user_question = "Our retail client is experiencing a 15% decline in same-store sales. What should we investigate?"

personas = {
    "McKinsey Partner (Strategy)": (
        "You are a McKinsey Senior Partner leading the Strategy & Corporate Finance practice. "
        "You think in terms of competitive dynamics, portfolio strategy, and value creation. "
        "Use structured, MECE frameworks and speak with authority."
    ),
    "McKinsey Associate (Operations)": (
        "You are a McKinsey Associate in the Operations practice. You focus on supply chain, "
        "store-level performance metrics, and operational efficiency. Be data-driven and suggest "
        "specific analyses to run."
    ),
    "McKinsey Expert (Digital & Analytics)": (
        "You are a McKinsey Digital & Analytics expert. You focus on digital channels, "
        "customer analytics, omnichannel strategy, and technology enablement. "
        "Recommend data-driven approaches and digital solutions."
    ),
}

print(f"Client Question: {user_question}")
print("=" * 60)

for persona_name, system_msg in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_question}
        ],
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")),
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n--- {persona_name} ---")
    print(response.choices[0].message.content)
    print()

Client Question: Our retail client is experiencing a 15% decline in same-store sales. What should we investigate?

--- McKinsey Partner (Strategy) ---
To address the 15% decline in same-store sales for your retail client, we can employ a structured, MECE (Mutually Exclusive, Collectively Exhaustive) framework to identify the root causes and potential solutions. We will categorize our investigation into three main areas: Market Dynamics, Customer Insights, and Operational Efficiency.

### 1. Market Dynamics
   - **Competitive Landscape**: 
     - Analyze competitors’ performance and strategies. Are they gaining market share? What promotions or innovations are they implementing?
     - Assess the entry of new competitors or substitutes that may be impacting sales.
   - **Economic Factors**: 
     - Evaluate macroeconomic indicators such as consumer confidence, unemployment rates, and disposable income trends that may affect spending behavior.
   - **Industry Trends**: 
     - Investigate

---
## Demo 5: Building a Basic LLM Pipeline

A pipeline chains multiple LLM calls together, where the output of one step becomes
the input of the next. This pattern is fundamental to agentic systems and mirrors how
structured analysis works -- first synthesize raw research, then extract structured
insights for client delivery.

In [14]:
# Demo 5 - Building a Basic LLM Pipeline

client = openai.OpenAI()

def call_llm(system_message, user_message, temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")), max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))):
    """Helper function to make an LLM call with a system and user message."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# --- Pipeline: Summarize Industry Research -> Extract Key Insights ---

industry_research = """
The global management consulting market is undergoing a fundamental transformation driven by 
generative AI and advanced analytics. McKinsey's own research indicates that up to 70% of 
business activities could be automated with current technology. Traditional strategy engagements 
that once required weeks of analyst research can now be augmented with AI-driven market analysis 
in days. However, the human element remains critical -- clients value the judgment, relationship 
management, and change leadership that experienced consultants provide. The shift is creating 
new service lines around AI implementation, responsible AI governance, and digital capability 
building. Firms that successfully integrate AI into their delivery model are seeing 30-40% 
improvements in engagement efficiency while maintaining or improving quality scores. The key 
challenges include managing data confidentiality across client engagements, ensuring AI outputs 
meet McKinsey's quality standards, building consultant capabilities in prompt engineering and 
AI-augmented analysis, and maintaining the trusted advisor relationship in an AI-augmented world.
"""

print("PIPELINE STEP 1: Summarize Industry Research")
print("=" * 60)

summary = call_llm(
    system_message="You are a McKinsey research analyst. Summarize the given text in 2-3 sentences suitable for a partner briefing.",
    user_message=f"Summarize this industry research:\n\n{industry_research}"
)
print(summary)

print("\nPIPELINE STEP 2: Extract Key Insights for Client Deck")
print("=" * 60)

key_points = call_llm(
    system_message="You are a McKinsey engagement manager preparing a client presentation. Extract exactly 5 key insights as a numbered list, each suitable for a slide bullet point.",
    user_message=f"Extract the key insights from this summary:\n\n{summary}"
)
print(key_points)

print("\nPIPELINE RESULT: Complete output")
print("=" * 60)
print(f"Original research : {len(industry_research.split())} words")
print(f"Summary length    : {len(summary.split())} words")
print(f"\nKey Insights for Client Deck:\n{key_points}")

PIPELINE STEP 1: Summarize Industry Research
The global management consulting market is rapidly evolving due to generative AI and advanced analytics, with McKinsey's research suggesting that up to 70% of business activities could be automated. While AI enhances efficiency—improving engagement by 30-40%—the human element remains essential for client relationships and change leadership. Key challenges include managing data confidentiality, ensuring quality standards, and developing consultant skills in AI integration while preserving the trusted advisor role.

PIPELINE STEP 2: Extract Key Insights for Client Deck
1. The global management consulting market is evolving, with generative AI and advanced analytics poised to automate up to 70% of business activities.

2. AI implementation can enhance efficiency, leading to a 30-40% improvement in client engagement.

3. Despite technological advancements, the human element is crucial for maintaining strong client relationships and effective cha

---
## Demo 6: Embeddings and Vector Representations

Embeddings convert text into dense numerical vectors that capture semantic meaning.
Two sentences with similar meaning will have similar embedding vectors, even if they
use different words. This is the foundation for:
- Semantic search across knowledge bases and past engagement documents
- Clustering client industries and market research by topic
- Similarity-based retrieval for RAG-powered assistants

**Key insight:** Embeddings capture *meaning*, not just keywords.
"Revenue growth decelerated in Q3" and "Sales momentum slowed in the third quarter"
will have very similar vectors despite using completely different words.

*Embeddings are covered hands-on in the Day 3 RAG sessions.*

---
## Demo 7: How LLMs Support Reasoning and Agent Behavior

LLMs exhibit several emergent capabilities that make them suitable as the reasoning
core of agentic systems. This demo showcases four of those capabilities:

1. **Task Decomposition** -- breaking a complex problem into MECE workstreams
2. **Planning** -- generating a step-by-step engagement plan
3. **Self-Reflection** -- critiquing and improving its own output
4. **Tool Selection** -- choosing the right analytical tool for a given request

These are the same capabilities you will wire together in Day 2 when you build
multi-step agents.

In [15]:
# Demo 7 - How LLMs Support Reasoning and Agent Behavior

client = openai.OpenAI()

# --- Capability 1: Task Decomposition (MECE Breakdown) ---
print("CAPABILITY 1: Task Decomposition (MECE Breakdown)")
print("=" * 60)

decomposition_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey engagement manager. Break down client problems into MECE (Mutually Exclusive, Collectively Exhaustive) workstreams with numbered sub-tasks."},
        {"role": "user", "content": "Our pharmaceutical client wants to evaluate entering the biosimilars market in Europe. Structure the key workstreams for this engagement."}
    ],
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)
print(decomposition_response.choices[0].message.content)

# --- Capability 2: Planning (Engagement Plan) ---
print("\n\nCAPABILITY 2: Planning -- Generating an Engagement Plan")
print("=" * 60)

planning_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": (
            "You are a McKinsey project manager. Given an engagement goal, output a numbered "
            "week-by-week plan of concrete actions. Each step should be a single deliverable. "
            "Format: 'Week N: [DELIVERABLE] -- description'"
        )},
        {"role": "user", "content": "Goal: Execute a 6-week due diligence for a private equity client evaluating a $500M acquisition of a B2B SaaS company."}
    ],
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")),
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)
print(planning_response.choices[0].message.content)

# --- Capability 3: Self-Reflection (Quality Review) ---
print("\n\nCAPABILITY 3: Self-Reflection -- Reviewing and Improving Analysis")
print("=" * 60)

# First, generate a deliberately brief answer
initial_answer = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "What are the key drivers of customer churn in the telecom industry?"}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "40"))
)
brief = initial_answer.choices[0].message.content
print(f"Initial (brief) answer: {brief}")

# Now ask the model to reflect on and improve its own answer -- like a partner review
reflection = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are a McKinsey Senior Partner reviewing an associate's work. Evaluate the answer below for completeness, MECE structure, and analytical rigor, then provide an improved version."},
        {"role": "user", "content": f"Original question: What are the key drivers of customer churn in the telecom industry?\n\nAssociate's draft: {brief}\n\nProvide your critique and an improved, partner-ready answer."}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "250"))
)
print(f"\nPartner review & improvement:\n{reflection.choices[0].message.content}")

# --- Capability 4: Tool Selection (Consulting Tools) ---
print("\n\nCAPABILITY 4: Tool Selection -- Choosing the Right Analytical Tool")
print("=" * 60)

tool_prompt = """You have access to these McKinsey analytical tools:
1. market_database -- for market size, growth rates, and competitive landscape data
2. financial_model -- for DCF, LBO, and valuation calculations
3. survey_platform -- for customer/employee surveys and NPS analysis
4. expert_network -- for scheduling calls with industry experts
5. benchmarking_tool -- for comparing client KPIs against industry benchmarks

For each request, respond with the tool you would select and why. Format: TOOL: [name] | REASON: [explanation]"""

tool_selection = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": tool_prompt},
        {"role": "user", "content": "What is the total addressable market for electric vehicle charging infrastructure in North America?"}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)
print(f"Request: 'TAM for EV charging infrastructure in North America'")
print(f"Decision: {tool_selection.choices[0].message.content}")

tool_selection2 = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": tool_prompt},
        {"role": "user", "content": "Is the target company's EBITDA margin competitive relative to peers in the specialty chemicals sector?"}
    ],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100"))
)
print(f"\nRequest: 'Is target EBITDA margin competitive vs specialty chemicals peers?'")
print(f"Decision: {tool_selection2.choices[0].message.content}")

CAPABILITY 1: Task Decomposition (MECE Breakdown)
To evaluate the potential entry of our pharmaceutical client into the biosimilars market in Europe, we can break down the engagement into the following MECE workstreams:

### Workstream 1: Market Analysis
1. **Market Size and Growth Potential**
   - 1.1: Analyze current market size of biosimilars in Europe.
   - 1.2: Assess historical growth rates and future projections.
   - 1.3: Identify key drivers of market growth (e.g., regulatory changes, cost savings).
  
2. **Competitive Landscape**
   - 2.1: Identify key competitors in the European biosimilars market.
   - 2.2: Analyze competitor market shares and product portfolios.
   - 2.3: Evaluate competitor strategies (e.g., pricing, distribution, partnerships).

3. **Regulatory Environment**
   - 3.1: Review European Medicines Agency (EMA) guidelines for biosimilars.
   - 3.2: Assess the approval process and timelines for biosimilars.
   - 3.3: Identify potential regulatory hurdles and r

---
## Summary

In these demos you saw the foundational skills for working with LLMs programmatically:

1. **API Connections** -- How to initialize the OpenAI client and make chat completion requests.
2. **Tokenization** -- How text is converted to tokens, why token counts matter for cost and context window limits.
3. **Model Parameters** -- How `temperature`, `top_p`, and `max_tokens` influence the style, creativity, and length of LLM outputs.
4. **System Messages & Personas** -- How the system message acts as a powerful control lever to change LLM behavior.
5. **LLM Pipelines** -- How to chain multiple LLM calls so the output of one step feeds into the next.
6. **Embeddings** -- How text is converted to vectors that capture semantic meaning (foundation for RAG on Day 3).
7. **LLM Reasoning** -- How LLMs exhibit task decomposition, planning, self-reflection, and tool selection.

**Next:** Open `D1S1_exercises.ipynb` to build a conversational agent yourself.